#<h1 align="center">**LMF Interactive STATS**</h1>




<div align="justify">

This is an interactive UMAP plot where you can easily explore all the processed samples from the Low Methane Forages project.

First go to **File** → **"Save a copy in Drive"**. This will create a copy of the notebook in your Google Drive. You can then edit the notebook and explore the data using the interactive controls.

To activate the visualization options, click the run_button.jpg button located at the top of the panel. Then, explore the different  functional groups. Once you have adjusted the plot to your preference, you can export it by clicking the three_dots_button.jpg button in the upper-right corner of the scatter plot.

**Advanced options**: You can customize the benchmark visualization by adding one or more id_lab values to the list in the second cell of the **Plot code** section. Benchmark samples will be highlighted in **purple**.

To highlight one or more samples of interest, add their **id** values to the target list. These samples will be displayed in **red**.

Don't know the id of your sample of interest? Explore the complete dataset [here.](https://github.com/maurope/lmf/blob/main/data/2026_08_20_database_categories_quartiles__visualization_public/compiled_categories_quartiles.csv)

</div>


# 1.0 Import libraries

In [1]:
import numpy as np
import pandas as pd
import altair as alt

# 2.0 Data load

In [2]:
url = "https://raw.githubusercontent.com/maurope/lmf/main/data/2026_08_20_database_categories_quartiles__visualization_public/compiled_categories_quartiles.csv"
df = pd.read_csv(url)
df

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,...,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,...,15.17,18.02,46.12,59.25,Medium,Not High,Category_2,81.0,Q1,58.0
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,...,12.98,16.25,46.61,58.55,Medium,Not High,Category_2,88.0,Q1,66.0
2,F24-3472,CIAT-12318,1,203.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,94.10,11.08,...,13.99,16.96,52.27,56.88,Medium,Not High,Category_2,148.0,Q2,78.0
3,F24-3427,CIAT-1257,1,113.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,2,92.72,9.52,...,15.71,17.60,46.32,58.95,Medium,Not High,Category_2,84.0,Q1,62.0
4,F24-3473,CIAT-13575,1,205.0,Genetic_bank,Desmodium incanum,Herbaceous_legumes,2,94.14,11.52,...,13.60,16.23,42.43,42.85,Low,Not High,Category_2,46.0,Q3,224.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Medium,Not High,Category_2,151.0,NaN,NaN
689,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,1.0,NaN,NaN
690,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,High,Not High,Category_4,36.0,NaN,NaN
691,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,55.0,NaN,NaN


# 3.0 Cake plots

## 3.1 Functional groups

In [21]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import re
import json
import pandas as pd
import altair as alt

# Disable Altair's default 5000-row safety limit, in case df is large.
alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, with at least the columns
# "functional_group", "subset", "lmf_category", "quartile", and
# "id_lab" (formatted like "F24-0001", where "24" -> year 2024).
# ============================================================

points_all = df.copy()

# Only these three functional groups are relevant here — same
# restriction used in the scatterplot.
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

# Same color scheme used across PCA / UMAP / scatterplot, for
# visual consistency across the whole toolkit.
functional_group_order = ["Herbaceous_legumes", "Grasses", "Shrub_Trees"]
functional_group_colors = ["green", "cornflowerblue", "orange"]

# ============================================================
# Processing year — extracted from id_lab's 2-digit code right
# before the dash (e.g. "F24-0001" -> "24" -> 2024). Used here only
# to build the dropdown's list of options in Python; the actual
# live filtering re-derives the same year from id_lab directly in
# the browser (see year_calc_expr below), so it works regardless of
# which rows are currently visible.
# ============================================================


def extract_year(id_lab):
    match = re.search(r"(\d{2})-", str(id_lab))
    return f"20{match.group(1)}" if match else None


points_all["_year"] = points_all["id_lab"].apply(extract_year)
year_options = ["All"] + sorted(points_all["_year"].dropna().unique().tolist())

# Vega expression that reproduces the same extraction logic
# (2 digits immediately before the first "-") directly on id_lab,
# so it's recalculated live for whatever rows are in view.
year_calc_expr = (
    "'20' + slice(datum.id_lab, indexof(datum.id_lab, '-') - 2, indexof(datum.id_lab, '-'))"
)

# ============================================================
# Dropdown options for subset / category / year
# ============================================================

subset_options = ["All"] + sorted(points_all["subset"].astype(str).unique().tolist())
category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())

# ============================================================
# Interactive controls
# ============================================================

sel_subset = alt.param(
    name="sel_subset",
    value="All",
    bind=alt.binding_select(options=subset_options, name="Subset: ")
)

sel_category = alt.param(
    name="sel_category",
    value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)

sel_year = alt.param(
    name="sel_year",
    value="All",
    bind=alt.binding_select(options=year_options, name="Processing year: ")
)

filter_expr = (
    "(sel_subset == 'All' || toString(datum.subset) == sel_subset) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    f"(sel_year == 'All' || ({year_calc_expr}) == sel_year)"
)

# ============================================================
# Title row — shows the current filter selection and total n,
# recalculated live in the browser via aggregation (no precomputed
# combinations needed, since this is just a row count).
# ============================================================

title_chart = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n="count()")
    .transform_calculate(
        label=(
            "'Functional Group Composition'"
            " + (sel_subset == 'All' ? '' : '  |  Subset: ' + sel_subset)"
            " + (sel_category == 'All' ? '' : '  |  Category: ' + sel_category)"
            " + (sel_year == 'All' ? '' : '  |  Year: ' + sel_year)"
        )
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=500, height=30)
)

# ============================================================
# Pie chart — percentage of samples per functional group,
# recalculated live from whatever subset/category/year is
# currently selected.
# ============================================================

pie_base = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(count="count()", groupby=["functional_group"])
    .transform_joinaggregate(total="sum(count)")
    .transform_calculate(percentage="datum.count / datum.total * 100")
    .transform_calculate(label_top="format(datum.percentage, '.1f') + '%'")
    .transform_calculate(label_bottom="'N = ' + datum.count")
    # Cumulative sum (in the same descending-by-count order used
    # visually) computed BEFORE any hover filtering, so each slice's
    # angular position is fixed and independent of which rows remain
    # visible afterward.
    .transform_window(
        cum_count="sum(count)",
        sort=[{"field": "count", "order": "descending"}],
        frame=[None, 0]
    )
    .transform_calculate(theta_start="(datum.cum_count - datum.count) / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_end="datum.cum_count / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_mid="(datum.theta_start + datum.theta_end) / 2")
)

FULL_CIRCLE_SCALE = alt.Scale(domain=[0, 2 * 3.14159265358979])

hover = alt.selection_point(
    fields=["functional_group"],
    on="pointerover",
    clear="pointerout",
    empty="all",
    name="hover"
)

base_slices = (
    pie_base
    .mark_arc(
        innerRadius=145,
        outerRadius=205,
        padAngle=0.015,
        cornerRadius=0,
        stroke="white",
        strokeWidth=2
    )
    .encode(
        theta=alt.Theta("theta_start:Q", scale=FULL_CIRCLE_SCALE, title=None),
        theta2=alt.Theta2("theta_end:Q"),
        color=alt.Color(
            "functional_group:N",
            scale=alt.Scale(domain=functional_group_order, range=functional_group_colors),
            legend=alt.Legend(title="Functional Group", symbolSize=100)
        ),
        opacity=alt.condition(hover, alt.value(1), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("functional_group:N", title="Functional group"),
            alt.Tooltip("count:Q", title="N samples"),
            alt.Tooltip("percentage:Q", title="Percentage", format=".1f")
        ]
    )
    .add_params(hover)
)

label_top_layer = (
    pie_base
    .mark_text(radius=240, dy=-8, fontSize=16, fontWeight="bold")
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_top:N"),
        color=alt.value("#333333")
    )
)

label_bottom_layer = (
    pie_base
    .mark_text(radius=240, dy=12, fontSize=12)
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_bottom:N"),
        color=alt.value("#666666")
    )
)

# Center label inside the donut hole, showing the overall total.
center_total = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n="count()")
    .transform_calculate(center_label="'Total\\nN = ' + datum.n")
    .mark_text(fontSize=20, fontWeight="bold", color="#333333", lineBreak="\\n", align="center")
    .encode(text="center_label:N")
)

pie_chart = (
    (base_slices + label_top_layer + label_bottom_layer + center_total)
    .properties(width=520, height=520)
)

# ============================================================
# Final layout — controls above, then title, then the chart
# ============================================================

final_plot = (
    alt.vconcat(title_chart, pie_chart)
    .add_params(sel_subset, sel_category, sel_year)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Functional Group Pie Chart</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
    /* Smooth hover animation for the donut slices */
    #vis path {{
      transition: opacity 0.2s ease;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "functional_group_pie.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "functional_group_pie.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "functional_group_pie.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("functional_group_pie.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("interactive_pie_chart_functional_groups.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: interactive_pie_chart_functional_groups.html")

Done: interactive_pie_chart_functional_groups.html


## 3.2 Tax name

In [26]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import re
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, with at least the columns
# "tax_name", "subset", "lmf_category", and "id_lab" (formatted
# like "F24-0001", where "24" -> year 2024).
# ============================================================

points_all = df.copy()
points_all = points_all[points_all["tax_name"].notna()].copy()

# Only these three functional groups are relevant — same restriction
# used everywhere else in the toolkit.
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

# How many individual species to show before collapsing the rest
# into a single "Other species" slice. Tune this to taste.
TOP_N_SPECIES = 8

# ============================================================
# Processing year (same extraction logic as the functional-group
# pie) — "F24-0001" -> "2024".
# ============================================================


def extract_year(id_lab):
    match = re.search(r"(\d{2})-", str(id_lab))
    return f"20{match.group(1)}" if match else None


points_all["_year"] = points_all["id_lab"].apply(extract_year)
year_options = ["All"] + sorted(points_all["_year"].dropna().unique().tolist())

year_calc_expr = (
    "'20' + slice(datum.id_lab, indexof(datum.id_lab, '-') - 2, indexof(datum.id_lab, '-'))"
)

# ============================================================
# Dropdown options
# ============================================================

subset_options = ["All"] + sorted(points_all["subset"].astype(str).unique().tolist())
category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())
functional_group_options = ["All"] + sorted(points_all["functional_group"].dropna().unique().tolist())

# ============================================================
# Interactive controls
# ============================================================

sel_subset = alt.param(
    name="sel_subset", value="All",
    bind=alt.binding_select(options=subset_options, name="Subset: ")
)
sel_category = alt.param(
    name="sel_category", value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)
sel_group = alt.param(
    name="sel_group", value="All",
    bind=alt.binding_select(options=functional_group_options, name="Functional group: ")
)
sel_year = alt.param(
    name="sel_year", value="All",
    bind=alt.binding_select(options=year_options, name="Processing year: ")
)

filter_expr = (
    "(sel_subset == 'All' || toString(datum.subset) == sel_subset) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    "(sel_group == 'All' || datum.functional_group == sel_group) && "
    f"(sel_year == 'All' || ({year_calc_expr}) == sel_year)"
)

# ============================================================
# Title — current filter selection + a live species/accession count
# ============================================================

title_chart = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n_accessions="count()", n_species="distinct(tax_name)")
    .transform_calculate(
        label=(
            "'Species Composition'"
            " + (sel_group == 'All' ? '' : '  |  Group: ' + sel_group)"
            " + (sel_subset == 'All' ? '' : '  |  Subset: ' + sel_subset)"
            " + (sel_category == 'All' ? '' : '  |  Category: ' + sel_category)"
            " + (sel_year == 'All' ? '' : '  |  Year: ' + sel_year)"
        )
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=520, height=30)
)

# ============================================================
# Pie chart — accessions per species, top N individually shown,
# the rest collapsed into "Other species". All recalculated live
# from whatever filters are currently active.
# ============================================================

pie_base = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    # Step 1: accessions per species
    .transform_aggregate(species_count="count()", groupby=["tax_name"])
    # Step 2: give each species a strictly unique sequential number,
    # ordered by accession count (highest first). Using row_number()
    # instead of rank() matters here: rank() gives EVERY tied species
    # (e.g. several species that all have exactly 1 accession) the
    # SAME rank number, so a whole tied group could all slip under
    # the TOP_N threshold at once instead of being collapsed into
    # "Other species" — that was producing dozens of near-invisible
    # slivers instead of one clean "Other" slice, which is what was
    # showing up as gaps in the donut.
    .transform_window(
        row_number="row_number()",
        sort=[
            {"field": "species_count", "order": "descending"},
            {"field": "tax_name", "order": "ascending"}
        ]
    )
    # Step 3: keep the top N species by name, collapse the rest
    .transform_calculate(
        species_group=f"datum.row_number <= {TOP_N_SPECIES} ? datum.tax_name : 'Other species'"
    )
    # Step 4: re-aggregate by that bucketed label — sum(species_count)
    # gives the accession total per slice, count() gives how many
    # distinct species got folded into that slice (relevant for
    # "Other species").
    .transform_aggregate(
        count="sum(species_count)",
        n_species_in_slice="count()",
        groupby=["species_group"]
    )
    .transform_joinaggregate(total="sum(count)")
    .transform_calculate(percentage="datum.count / datum.total * 100")
    .transform_calculate(label_top="format(datum.percentage, '.1f') + '%'")
    .transform_calculate(label_bottom="'N = ' + datum.count")
    .transform_window(
        cum_count="sum(count)",
        sort=[{"field": "count", "order": "descending"}],
        frame=[None, 0]
    )
    .transform_calculate(theta_start="(datum.cum_count - datum.count) / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_end="datum.cum_count / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_mid="(datum.theta_start + datum.theta_end) / 2")
)

FULL_CIRCLE_SCALE = alt.Scale(domain=[0, 2 * 3.14159265358979])

hover = alt.selection_point(
    fields=["species_group"],
    on="pointerover",
    clear="pointerout",
    empty="all",
    name="hover"
)

base_slices = (
    pie_base
    .mark_arc(
        innerRadius=145,
        outerRadius=205,
        padAngle=0.006,
        cornerRadius=0,
        stroke="white",
        strokeWidth=1
    )
    .encode(
        theta=alt.Theta("theta_start:Q", scale=FULL_CIRCLE_SCALE, title=None),
        theta2=alt.Theta2("theta_end:Q"),
        color=alt.Color(
            "species_group:N",
            sort=alt.SortField("count", order="descending"),
            scale=alt.Scale(scheme="tableau20"),
            legend=alt.Legend(title="Species (top {} + other)".format(TOP_N_SPECIES), symbolSize=90, labelLimit=220)
        ),
        opacity=alt.condition(hover, alt.value(1), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("species_group:N", title="Species"),
            alt.Tooltip("count:Q", title="N accessions"),
            alt.Tooltip("percentage:Q", title="Percentage", format=".1f"),
            alt.Tooltip("n_species_in_slice:Q", title="Species in this slice")
        ]
    )
    .add_params(hover)
)

label_top_layer = (
    pie_base
    .mark_text(radius=240, dy=-8, fontSize=15, fontWeight="bold")
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_top:N"),
        color=alt.value("#333333")
    )
)

label_bottom_layer = (
    pie_base
    .mark_text(radius=240, dy=12, fontSize=11)
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_bottom:N"),
        color=alt.value("#666666")
    )
)

# Center label — total distinct species AND total accessions,
# always reflecting the current filters (not the top-N grouping).
center_total = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n_accessions="count()", n_species="distinct(tax_name)")
    .transform_calculate(
        center_label="datum.n_species + ' species\\n' + datum.n_accessions + ' accessions'"
    )
    .mark_text(fontSize=18, fontWeight="bold", color="#333333", lineBreak="\\n", align="center")
    .encode(text="center_label:N")
)

pie_chart = (
    (base_slices + label_top_layer + label_bottom_layer + center_total)
    .properties(width=520, height=520)
)

# ============================================================
# Final layout
# ============================================================

final_plot = (
    alt.vconcat(title_chart, pie_chart)
    .add_params(sel_subset, sel_category, sel_group, sel_year)
    .configure_legend(titleFontSize=15, labelFontSize=12)
    .configure_view(strokeWidth=0)
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Species Pie Chart</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
    #vis path {{
      transition: opacity 0.2s ease;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "species_pie.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "species_pie.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "species_pie.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("species_pie.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("interactive_species_pie.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: interactive_species_pie.html")

Done: interactive_species_pie.html


# 4.0 Counting Banner

In [4]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: this piece is built with plain HTML/CSS (Flexbox), not
# Vega-Lite — the nested box layout (small boxes under big ones,
# plus a side box matching their combined height) is a layout
# problem, not a data-encoding one, and CSS handles it far more
# reliably than trying to force it into a chart grammar.
#
# This assumes `df` already exists, with "functional_group" and
# "lmf_category" columns.

import pandas as pd

points_all = df.copy()

valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

functional_group_order = ["Herbaceous_legumes", "Grasses", "Shrub_Trees"]
functional_group_colors = {
    "Herbaceous_legumes": "#008000",     # green
    "Grasses": "#6495ED",                # cornflowerblue
    "Shrub_Trees": "#FFA500",            # orange
}

# The 5 LMF categories shown as small boxes under each group.
# Adjust this list if your actual category set is different —
# any category not present in the data for a given group just
# shows 0.
CATEGORY_LABELS = ["Category_1a", "Category_1b", "Category_2", "Category_3", "Category_4"]
NEUTRAL_GRAY = "#B0B0B0"
TOTAL_BOX_COLOR = "purple"


def blend(hex_color, target_hex, factor):
    """Blend hex_color toward target_hex by `factor` (0 = hex_color, 1 = target_hex)."""
    c = hex_color.lstrip("#")
    t = target_hex.lstrip("#")
    c_rgb = [int(c[i:i + 2], 16) for i in (0, 2, 4)]
    t_rgb = [int(t[i:i + 2], 16) for i in (0, 2, 4)]
    blended = [round(c_rgb[i] + (t_rgb[i] - c_rgb[i]) * factor) for i in range(3)]
    return "#{:02x}{:02x}{:02x}".format(*blended)


def shade_palette(base_hex, n):
    """n distinct shades of base_hex, from darker to lighter (base color in the middle)."""
    factors = [(-0.35, "#000000"), (-0.15, "#000000"), (0, None), (0.20, "#ffffff"), (0.40, "#ffffff")]
    shades = []
    for f, target in factors[:n]:
        if target is None:
            shades.append(base_hex)
        else:
            shades.append(blend(base_hex, target, abs(f)))
    return shades

# ============================================================
# Compute the numbers
# ============================================================

group_totals = points_all["functional_group"].value_counts().to_dict()

category_counts = {
    group: (
        points_all.loc[points_all["functional_group"] == group, "lmf_category"]
        .value_counts()
        .to_dict()
    )
    for group in functional_group_order
}

grand_total = len(points_all)

# ============================================================
# Build the HTML
# ============================================================

def render_group_column(group):
    color = functional_group_colors[group]
    total = group_totals.get(group, 0)
    group_pct = (total / grand_total * 100) if grand_total else 0
    shades = shade_palette(color, len(CATEGORY_LABELS))

    sub_boxes_html = ""
    for cat, shade_color in zip(CATEGORY_LABELS, shades):
        count = category_counts.get(group, {}).get(cat, 0)
        cat_pct = (count / total * 100) if total else 0
        short_label = cat.replace("Category_", "")
        sub_boxes_html += f"""
        <div class="sub-box" style="background: {shade_color};">
          <div class="sub-box-label">{short_label}</div>
          <div class="sub-box-count">{count}</div>
          <div class="sub-box-percent">{cat_pct:.1f}%</div>
        </div>"""

    return f"""
    <div class="group-col">
      <div class="group-box" style="background: {color};">
        <div class="group-name">{group}</div>
        <div class="group-count">{total}</div>
        <div class="group-percent">{group_pct:.1f}%</div>
      </div>
      <div class="sub-boxes-row">
        {sub_boxes_html}
      </div>
    </div>"""


groups_html = "".join(render_group_column(g) for g in functional_group_order)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Functional Group Banner</title>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      justify-content: center;
      padding: 30px;
    }}

    .dashboard {{
      display: flex;
      align-items: stretch;
      gap: 20px;
    }}

    .groups {{
      display: flex;
      gap: 16px;
    }}

    .total-box {{
      width: 220px;
      background: {TOTAL_BOX_COLOR};
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      color: white;
      transition: background 0.2s ease;
    }}
    .total-label {{
      font-size: 15px;
      font-weight: bold;
      margin-bottom: 6px;
    }}
    .total-count {{
      font-size: 48px;
      font-weight: bold;
    }}
    .total-percent {{
      font-size: 48px;
      font-weight: bold;
      display: none;
    }}
    .total-box:hover .total-count {{
      display: none;
    }}
    .total-box:hover .total-percent {{
      display: block;
    }}
    /* Hovering ANY box (group, sub-box, or total) grays out every
       OTHER box in the whole dashboard — regardless of type or
       group. Only the exact box under the cursor keeps its color. */
    .dashboard:hover .group-box:not(:hover),
    .dashboard:hover .sub-box:not(:hover),
    .dashboard:hover .total-box:not(:hover) {{
      background: {NEUTRAL_GRAY} !important;
    }}

    .group-col {{
      display: flex;
      flex-direction: column;
      gap: 10px;
    }}

    .group-box {{
      width: 300px;
      height: 130px;
      border-radius: 0;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      color: white;
      transition: background 0.2s ease;
    }}
    /* (Cross-box graying handled by the single global rule above.) */
    .group-name {{
      font-size: 15px;
      font-weight: bold;
      margin-bottom: 6px;
    }}
    .group-count {{
      font-size: 40px;
      font-weight: bold;
    }}
    .group-percent {{
      font-size: 40px;
      font-weight: bold;
      display: none;
    }}
    .group-box:hover .group-count {{
      display: none;
    }}
    .group-box:hover .group-percent {{
      display: block;
    }}

    .sub-boxes-row {{
      display: flex;
      gap: 6px;
    }}

    .sub-box {{
      flex: 1;
      height: 70px;
      border-radius: 0;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      color: white;
    }}
    .sub-box-label {{
      font-size: 11px;
      font-weight: bold;
    }}
    .sub-box-count {{
      font-size: 18px;
      font-weight: bold;
    }}
    .sub-box-percent {{
      font-size: 15px;
      font-weight: bold;
      display: none;
    }}
    .sub-box:hover .sub-box-count {{
      display: none;
    }}
    .sub-box:hover .sub-box-percent {{
      display: block;
    }}
  </style>
</head>
<body>

  <div class="dashboard">
    <div class="total-box">
      <div class="total-label">Total sampled</div>
      <div class="total-count">{grand_total}</div>
      <div class="total-percent">100.0%</div>
    </div>

    <div class="groups">
      {groups_html}
    </div>
  </div>

</body>
</html>
"""

with open("interactive_banner.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: interactive_banner.html")

Done: interactive_banner.html


# 5.0 Bars

In [18]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, with at least the columns
# "functional_group" and "lmf_category" (same data used for the pie).
# ============================================================

points_all = df.copy()

valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

functional_group_order = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]

# Categories are read directly from the data, so this adapts to
# whatever LMF categories your real dataset actually contains.
found_categories = sorted(points_all["lmf_category"].dropna().unique().tolist())

# Taxon names, same idea — read directly from the data.
tax_name_order = sorted(points_all["tax_name"].dropna().unique().tolist())

# Fixed, explicit color per category NAME (matching the reference
# image) — intentionally NOT positional/index-based, so a category
# being entirely absent from the data (or from a specific functional
# group / filter selection) never shifts the color of any other
# category.
CATEGORY_COLOR_MAP = {
        "Category_1a": "#E8A93B",    # gold
        "Category_1b": "#8BC63F",   # green
        "Category_2": "#A9CC93",    # light green
        "Category_3": "#1CADA8",   # teal
        "Category_4": "#E6E6E6",    # gray
}
FALLBACK_PALETTE = ["#B07AA1", "#E15759", "#76B7B2", "#FF9DA7", "#9C755F"]

# Preserve the canonical order first (only the categories actually
# present), then append any unexpected category names not in the
# fixed map, giving those a fallback color.
category_order = (
    [c for c in CATEGORY_COLOR_MAP if c in found_categories]
    + [c for c in found_categories if c not in CATEGORY_COLOR_MAP]
)

category_colors = {}
_fallback_i = 0
for c in category_order:
    if c in CATEGORY_COLOR_MAP:
        category_colors[c] = CATEGORY_COLOR_MAP[c]
    else:
        category_colors[c] = FALLBACK_PALETTE[_fallback_i % len(FALLBACK_PALETTE)]
        _fallback_i += 1

# ============================================================
# Dropdown filters — same style as the rest of the toolkit
# (single-select "All" dropdowns, not checkboxes).
# ============================================================

group_options = ["All"] + functional_group_order
category_options = ["All"] + category_order
tax_name_options = ["All"] + tax_name_order

sel_group = alt.param(
    name="sel_group",
    value="All",
    bind=alt.binding_select(options=group_options, name="Functional group: ")
)

sel_category = alt.param(
    name="sel_category",
    value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)

sel_tax = alt.param(
    name="sel_tax",
    value="All",
    bind=alt.binding_select(options=tax_name_options, name="Taxon name: ")
)

filter_expr = (
    "(sel_group == 'All' || datum.functional_group == sel_group) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    "(sel_tax == 'All' || datum.tax_name == sel_tax)"
)

# ============================================================
# Title
# ============================================================

title_chart = (
    alt.Chart(pd.DataFrame({"_dummy": [1]}))
    .transform_calculate(
        label=(
            "sel_group == 'All' "
            "? 'Number of accessions by Functional Group' "
            ": 'Number of accessions by Category'"
        )
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=550, height=30)
)

# ============================================================
# Grouped bar chart — no legend; each bar carries its own
# category label + count directly above it.
# ============================================================

bar_base = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(count="count()", groupby=["functional_group", "lmf_category"])
    .transform_calculate(short_cat="replace(datum.lmf_category, 'Category_', '')")
    .transform_calculate(bar_label="datum.short_cat + ': ' + datum.count")
)

bars = (
    bar_base
    .mark_bar()
    .encode(
        x=alt.X(
            "functional_group:N",
            sort=functional_group_order,
            title="Functional Group",
            axis=alt.Axis(labelAngle=0, labelPadding=8)
        ),
        xOffset=alt.XOffset(
            "lmf_category:N",
            sort=category_order,
            scale=alt.Scale(paddingInner=0.15)
        ),
        y=alt.Y("count:Q", title="Number of accessions", axis=alt.Axis(grid=False)),
        color=alt.Color(
            "lmf_category:N",
            sort=category_order,
            scale=alt.Scale(domain=category_order, range=[category_colors[c] for c in category_order]),
            legend=None
        ),
        tooltip=[
            alt.Tooltip("functional_group:N", title="Functional group"),
            alt.Tooltip("lmf_category:N", title="LMF category"),
            alt.Tooltip("count:Q", title="N accessions")
        ]
    )
)

bar_chart = bars.properties(width=550, height=420)

# ============================================================
# Final layout
# ============================================================

final_plot = (
    alt.vconcat(title_chart, bar_chart)
    .add_params(sel_group, sel_category, sel_tax)
    .configure_axis(labelFontSize=13, titleFontSize=15)
    .configure_view(strokeWidth=0)
)

# ============================================================
# Save as interactive HTML with the dropdowns above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Priority Observations by Group and Category</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "priority_observations.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "priority_observations.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "priority_observations.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("priority_observations.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("interactive_bar_chart.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: interactive_bar_chart.html")

Done: interactive_bar_chart.html


# 6.0 Data explorer

In [6]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: this uses Tabulator (https://tabulator.info), a JS table
# library loaded via CDN — no Vega-Lite involved here, since this
# is a data-grid task, not a chart.
#
# This assumes `df` already exists (the same dataframe you've been
# using for the pie/banner/bar chart).

import json
import math
import pandas as pd

points_all = df.copy()

# ============================================================
# Decide, per column, what kind of filter makes sense:
# - Numeric columns  -> a "≥" number filter (type a number, see
#   rows with that value or higher)
# - Low-cardinality text/category columns (<= 20 unique values)
#   -> a dropdown ("select") filter
# - Everything else (e.g. free-text columns like tax_name/id)
#   -> a "contains" text filter
# ============================================================

MAX_CATEGORICAL_UNIQUE = 20

column_defs = []
for col in points_all.columns:
    series = points_all[col]
    is_numeric = pd.api.types.is_numeric_dtype(series)

    if is_numeric:
        column_defs.append({
            "title": col,
            "field": col,
            "headerFilter": "input",
            "headerFilterFunc": ">=",
            "headerFilterPlaceholder": "≥ value",
            "sorter": "number"
        })
    else:
        n_unique = series.dropna().nunique()
        if n_unique <= MAX_CATEGORICAL_UNIQUE:
            options = ["(All)"] + sorted(series.dropna().unique().tolist())
            column_defs.append({
                "title": col,
                "field": col,
                "headerFilter": "list",
                "headerFilterParams": {"values": options, "clearable": True},
                "headerFilterFunc": "=",
                "sorter": "string"
            })
        else:
            column_defs.append({
                "title": col,
                "field": col,
                "headerFilter": "input",
                "headerFilterFunc": "like",
                "headerFilterPlaceholder": "contains...",
                "sorter": "string"
            })

# ============================================================
# Data — NaN -> null, so it serializes cleanly to JSON
# ============================================================

records = points_all.where(pd.notnull(points_all), None).to_dict(orient="records")

data_json = json.dumps(records)
columns_json = json.dumps(column_defs)

# ============================================================
# Header accent colors for specific column groups
# ============================================================

COLUMN_COLORS = {
    "id": ("#EEF1FF", "#3B4FCC"),           # indigo — identifier
    "dm_percentage": ("#EAF7EE", "#2F855A"),      # green — nutrition composition
    "ash_dm": ("#EAF7EE", "#2F855A"),
    "om_percentage": ("#EAF7EE", "#2F855A"),
    "pc_percentage_dm": ("#EAF7EE", "#2F855A"),
    "adf_percentage_dm": ("#EAF7EE", "#2F855A"),
    "ndf_percentage_dm": ("#EAF7EE", "#2F855A"),
    "methane_intensity": ("#FFF3E6", "#B15C00"),  # orange — methane metrics
    "tddm": ("#FFF3E6", "#B15C00"),
    "lmf_category": ("#F3EAFB", "#7B2FBE"),       # purple — classification
}

column_color_css = "\n".join(
    f"""
    .tabulator-col[tabulator-field="{field}"] {{
      background: {bg} !important;
    }}
    .tabulator-col[tabulator-field="{field}"] .tabulator-col-title {{
      color: {fg} !important;
      font-weight: 700 !important;
    }}"""
    for field, (bg, fg) in COLUMN_COLORS.items()
)

# ============================================================
# Build the HTML
# ============================================================

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Data Explorer</title>
  <link href="https://unpkg.com/tabulator-tables@5.5.2/dist/css/tabulator.min.css" rel="stylesheet">
  <script src="https://unpkg.com/tabulator-tables@5.5.2/dist/js/tabulator.min.js"></script>
  <style>
    * {{
      box-sizing: border-box;
    }}
    body {{
      font-family: 'Segoe UI', -apple-system, system-ui, Roboto, sans-serif;
      background: #f6f7fb;
      padding: 32px;
      color: #2b2b33;
    }}
    .card {{
      background: white;
      border-radius: 14px;
      box-shadow: 0 2px 12px rgba(20, 20, 43, 0.06);
      padding: 24px 28px 28px;
      max-width: 1200px;
      margin: 0 auto;
    }}
    h2 {{
      margin: 0 0 4px;
      font-size: 22px;
      font-weight: 700;
      color: #1f1f2e;
    }}
    #meta {{
      color: #8a8a99;
      margin-bottom: 18px;
      font-size: 14px;
    }}
    #meta b {{
      color: #4a4a5a;
    }}
    #toolbar {{
      display: flex;
      gap: 10px;
      margin-bottom: 18px;
    }}
    #toolbar button {{
      padding: 9px 18px;
      font-size: 14px;
      font-weight: 600;
      border: none;
      border-radius: 8px;
      cursor: pointer;
      transition: transform 0.05s ease, box-shadow 0.15s ease;
    }}
    #btn-download-csv {{
      background: #4C6FFF;
      color: white;
      box-shadow: 0 2px 8px rgba(76, 111, 255, 0.35);
    }}
    #btn-download-csv:hover {{
      background: #3d5ce0;
    }}
    #btn-clear-filters {{
      background: #f1f1f5;
      color: #55555f;
    }}
    #btn-clear-filters:hover {{
      background: #e6e6ec;
    }}
    #toolbar button:active {{
      transform: scale(0.98);
    }}

    /* ---- Tabulator theme overrides: cleaner, less gray ---- */
    .tabulator {{
      border: none !important;
      background: transparent !important;
      font-size: 13.5px;
    }}
    .tabulator-header {{
      background: #fafafe !important;
      border-bottom: 2px solid #ececf3 !important;
    }}
    .tabulator-col {{
      background: #fafafe !important;
      border-right: none !important;
    }}
    .tabulator-col-title {{
      color: #3a3a45;
      font-weight: 600;
    }}
    .tabulator-row {{
      background: white !important;
      border-bottom: 1px solid #f0f0f5 !important;
    }}
    .tabulator-row:hover {{
      background: #f5f7ff !important;
    }}
    .tabulator-row.tabulator-row-even {{
      background: #fbfbfe !important;
    }}
    .tabulator-cell {{
      padding: 8px 10px !important;
    }}
    .tabulator-header-filter input,
    .tabulator-header-filter select {{
      border: 1px solid #e2e2ec !important;
      border-radius: 6px !important;
      padding: 4px 6px !important;
    }}
    .tabulator-footer {{
      background: #fafafe !important;
      border-top: 2px solid #ececf3 !important;
    }}
    .tabulator-paginator {{
      color: #55555f;
    }}
    .tabulator-page {{
      border-radius: 6px !important;
    }}
    .tabulator-page.active {{
      background: #4C6FFF !important;
      color: white !important;
      border-color: #4C6FFF !important;
    }}

    /* ---- Per-column header accent colors ---- */
    {column_color_css}
  </style>
</head>
<body>

  <div class="card">
    <h2>Data Explorer</h2>
    <div id="meta">Showing <b><span id="row-count">0</span></b> of <b>{len(records)}</b> rows — use the boxes under each column header to filter.</div>

    <div id="toolbar">
      <button id="btn-download-csv">⬇ Download filtered data (CSV)</button>
      <button id="btn-clear-filters">✕ Clear all filters</button>
    </div>

    <div id="table"></div>
  </div>

  <script type="text/javascript">
    const tableData = {data_json};
    const columnDefs = {columns_json};

    const table = new Tabulator("#table", {{
      data: tableData,
      columns: columnDefs,
      layout: "fitDataFill",
      height: "650px",
      pagination: true,
      paginationSize: 25,
      paginationSizeSelector: [10, 25, 50, 100, true],
    }});

    function updateRowCount() {{
      document.getElementById("row-count").textContent = table.getDataCount("active");
    }}

    table.on("tableBuilt", updateRowCount);
    table.on("dataFiltered", updateRowCount);

    // Build the CSV ourselves from the currently filtered ("active")
    // rows, so the download always matches exactly what's on screen.
    function toCsv(rows, columns) {{
      const fields = columns.map(c => c.field);
      const escape = (val) => {{
        if (val === null || val === undefined) return "";
        const s = String(val);
        if (/[",\\n]/.test(s)) {{
          return '"' + s.replace(/"/g, '""') + '"';
        }}
        return s;
      }};
      const header = fields.map(escape).join(",");
      const lines = rows.map(row => fields.map(f => escape(row[f])).join(","));
      return [header, ...lines].join("\\n");
    }}

    document.getElementById("btn-download-csv").addEventListener("click", function() {{
      const activeRows = table.getData("active");
      const csv = toCsv(activeRows, columnDefs);
      const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
      const url = URL.createObjectURL(blob);
      const link = document.createElement("a");
      link.href = url;
      link.download = "filtered_data.csv";
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      URL.revokeObjectURL(url);
    }});

    document.getElementById("btn-clear-filters").addEventListener("click", function() {{
      table.clearHeaderFilter();
    }});
  </script>
</body>
</html>
"""

with open("data_explorer.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: data_explorer.html")

Done: data_explorer.html


In [7]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: this uses Tabulator (https://tabulator.info), a JS table
# library loaded via CDN — no Vega-Lite involved here, since this
# is a data-grid task, not a chart.
#
# This assumes `df` already exists (the same dataframe you've been
# using for the pie/banner/bar chart).

import json
import math
import pandas as pd

points_all = df.copy()

# ============================================================
# Decide, per column, what kind of filter makes sense:
# - Numeric columns  -> a "≥" number filter (type a number, see
#   rows with that value or higher)
# - Low-cardinality text/category columns (<= 20 unique values)
#   -> a dropdown ("select") filter
# - Everything else (e.g. free-text columns like tax_name/id)
#   -> a "contains" text filter
# ============================================================

MAX_CATEGORICAL_UNIQUE = 20

column_defs = []
for col in points_all.columns:
    series = points_all[col]
    is_numeric = pd.api.types.is_numeric_dtype(series)

    if is_numeric:
        column_defs.append({
            "title": col,
            "field": col,
            "headerFilter": "input",
            "headerFilterFunc": ">=",
            "headerFilterPlaceholder": "≥ value",
            "sorter": "number"
        })
    else:
        n_unique = series.dropna().nunique()
        if n_unique <= MAX_CATEGORICAL_UNIQUE:
            options = ["(All)"] + sorted(series.dropna().unique().tolist())
            column_defs.append({
                "title": col,
                "field": col,
                "headerFilter": "list",
                "headerFilterParams": {"values": options, "clearable": True},
                "headerFilterFunc": "=",
                "sorter": "string"
            })
        else:
            column_defs.append({
                "title": col,
                "field": col,
                "headerFilter": "input",
                "headerFilterFunc": "like",
                "headerFilterPlaceholder": "contains...",
                "sorter": "string"
            })

# ============================================================
# Data — NaN -> null, so it serializes cleanly to JSON
# ============================================================

records = points_all.where(pd.notnull(points_all), None).to_dict(orient="records")

data_json = json.dumps(records)
columns_json = json.dumps(column_defs)

# ============================================================
# Header accent colors for specific column groups
# ============================================================

COLUMN_COLORS = {
    "id": ("#EEF1FF", "#3B4FCC"),           # indigo — identifier
    "dm_percentage": ("#EAF7EE", "#2F855A"),      # green — nutrition composition
    "ash_dm": ("#EAF7EE", "#2F855A"),
    "om_percentage": ("#EAF7EE", "#2F855A"),
    "pc_percentage_dm": ("#EAF7EE", "#2F855A"),
    "adf_percentage_dm": ("#EAF7EE", "#2F855A"),
    "ndf_percentage_dm": ("#EAF7EE", "#2F855A"),
    "methane_intensity": ("#FFF3E6", "#B15C00"),  # orange — methane metrics
    "tddm": ("#FFF3E6", "#B15C00"),
    "lmf_category": ("#F3EAFB", "#7B2FBE"),       # purple — classification
}

column_color_css = "\n".join(
    f"""
    .tabulator-col[tabulator-field="{field}"] {{
      background: {bg} !important;
    }}
    .tabulator-col[tabulator-field="{field}"] .tabulator-col-title {{
      color: {fg} !important;
      font-weight: 700 !important;
    }}"""
    for field, (bg, fg) in COLUMN_COLORS.items()
)

# ============================================================
# Build the HTML
# ============================================================

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Data Explorer</title>
  <link href="https://unpkg.com/tabulator-tables@5.5.2/dist/css/tabulator.min.css" rel="stylesheet">
  <script src="https://unpkg.com/tabulator-tables@5.5.2/dist/js/tabulator.min.js"></script>
  <style>
    * {{
      box-sizing: border-box;
    }}
    body {{
      font-family: 'Segoe UI', -apple-system, system-ui, Roboto, sans-serif;
      background: #f6f7fb;
      padding: 32px;
      color: #2b2b33;
    }}
    .card {{
      background: white;
      border-radius: 14px;
      box-shadow: 0 2px 12px rgba(20, 20, 43, 0.06);
      padding: 24px 28px 28px;
      max-width: 1400px;
      margin: 0 auto;
    }}
    h2 {{
      margin: 0 0 4px;
      font-size: 22px;
      font-weight: 700;
      color: #1f1f2e;
    }}
    #meta {{
      color: #8a8a99;
      margin-bottom: 18px;
      font-size: 14px;
    }}
    #meta b {{
      color: #4a4a5a;
    }}
    #toolbar {{
      display: flex;
      gap: 10px;
      margin-bottom: 18px;
    }}
    #toolbar button {{
      padding: 9px 18px;
      font-size: 14px;
      font-weight: 600;
      border: none;
      border-radius: 8px;
      cursor: pointer;
      transition: transform 0.05s ease, box-shadow 0.15s ease;
    }}
    #btn-download-csv {{
      background: #4C6FFF;
      color: white;
      box-shadow: 0 2px 8px rgba(76, 111, 255, 0.35);
    }}
    #btn-download-csv:hover {{
      background: #3d5ce0;
    }}
    #btn-clear-filters {{
      background: #f1f1f5;
      color: #55555f;
    }}
    #btn-clear-filters:hover {{
      background: #e6e6ec;
    }}
    #toolbar button:active {{
      transform: scale(0.98);
    }}

    /* ---- Tabulator theme overrides: cleaner, less gray ---- */
    .tabulator {{
      border: none !important;
      background: transparent !important;
      font-size: 13.5px;
    }}
    .tabulator-header {{
      background: #fafafe !important;
      border-bottom: 2px solid #ececf3 !important;
    }}
    .tabulator-col {{
      background: #fafafe !important;
      border-right: none !important;
    }}
    .tabulator-col-title {{
      color: #3a3a45;
      font-weight: 600;
    }}
    .tabulator-row {{
      background: white !important;
      border-bottom: 1px solid #f0f0f5 !important;
    }}
    .tabulator-row:hover {{
      background: #f5f7ff !important;
    }}
    .tabulator-row.tabulator-row-even {{
      background: #fbfbfe !important;
    }}
    .tabulator-cell {{
      padding: 8px 10px !important;
    }}
    .tabulator-header-filter input,
    .tabulator-header-filter select {{
      border: 1px solid #e2e2ec !important;
      border-radius: 6px !important;
      padding: 4px 6px !important;
    }}
    .tabulator-footer {{
      background: #fafafe !important;
      border-top: 2px solid #ececf3 !important;
    }}
    .tabulator-paginator {{
      color: #55555f;
    }}
    .tabulator-page {{
      border-radius: 6px !important;
    }}
    .tabulator-page.active {{
      background: #4C6FFF !important;
      color: white !important;
      border-color: #4C6FFF !important;
    }}

    /* ---- Per-column header accent colors ---- */
    {column_color_css}
  </style>
</head>
<body>

  <div class="card">
    <h2>Data Explorer</h2>
    <div id="meta">Showing <b><span id="row-count">0</span></b> of <b>{len(records)}</b> rows — use the boxes under each column header to filter.</div>

    <div id="toolbar">
      <button id="btn-download-csv">⬇ Download filtered data (CSV)</button>
      <button id="btn-clear-filters">✕ Clear all filters</button>
    </div>

    <div id="table"></div>
  </div>

  <script type="text/javascript">
    const tableData = {data_json};
    const columnDefs = {columns_json};

    const table = new Tabulator("#table", {{
      data: tableData,
      columns: columnDefs,
      layout: "fitDataFill",
      height: "650px",
      pagination: true,
      paginationSize: 25,
      paginationSizeSelector: [10, 25, 50, 100, true],
    }});

    function updateRowCount() {{
      document.getElementById("row-count").textContent = table.getDataCount("active");
    }}

    table.on("tableBuilt", updateRowCount);
    table.on("dataFiltered", updateRowCount);

    // Build the CSV ourselves from the currently filtered ("active")
    // rows, so the download always matches exactly what's on screen.
    function toCsv(rows, columns) {{
      const fields = columns.map(c => c.field);
      const escape = (val) => {{
        if (val === null || val === undefined) return "";
        const s = String(val);
        if (/[",\\n]/.test(s)) {{
          return '"' + s.replace(/"/g, '""') + '"';
        }}
        return s;
      }};
      const header = fields.map(escape).join(",");
      const lines = rows.map(row => fields.map(f => escape(row[f])).join(","));
      return [header, ...lines].join("\\n");
    }}

    document.getElementById("btn-download-csv").addEventListener("click", function() {{
      const activeRows = table.getData("active");
      const csv = toCsv(activeRows, columnDefs);
      const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
      const url = URL.createObjectURL(blob);
      const link = document.createElement("a");
      link.href = url;
      link.download = "filtered_data.csv";
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      URL.revokeObjectURL(url);
    }});

    document.getElementById("btn-clear-filters").addEventListener("click", function() {{
      table.clearHeaderFilter();
    }});
  </script>
</body>
</html>
"""

with open("data_explorer.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: data_explorer.html")

Done: data_explorer.html


# 7.0 Select Accesions

In [11]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: uses Tabulator (https://tabulator.info) via CDN for the
# table — this is a data-grid task, not a chart.
#
# This assumes `df` already exists, with (at least) the columns:
# functional_group, lmf_category, quartile, id, id_lab,
# ch4_percentage_in_gas_8h, ch4_percentage_in_gas_24h,
# methane_intensity, tddm, ch4_category, tddm_category,
# lmf_category_rank, quartile_rank.

import json
import pandas as pd

points_all = df.copy()

# Ignore any functional_group values other than these three.
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

# Columns shown in the results table/download, in this exact order.
DISPLAY_COLUMNS = [
    "id",
    "id_lab",
    "tax_name",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm",
    "ch4_category",
    "tddm_category",
    "lmf_category",
    "lmf_category_rank",
    "quartile",
    "quartile_rank",
]

# Columns used only for filtering (not displayed as their own
# table column, but kept in the row data so the filters can match
# against them).
FILTER_ONLY_COLUMNS = ["functional_group"]

missing = [c for c in DISPLAY_COLUMNS + FILTER_ONLY_COLUMNS if c not in points_all.columns]
if missing:
    raise ValueError(f"These expected columns are missing from df: {missing}")

# ============================================================
# Filter dropdown options
# ============================================================

functional_group_options = ["All"] + sorted(points_all["functional_group"].dropna().unique().tolist())
lmf_category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())
quartile_options = ["All"] + sorted(points_all["quartile"].dropna().unique().tolist())

# ============================================================
# Data — keep display columns + the filter-only columns, NaN -> null
# ============================================================

data_df = points_all[DISPLAY_COLUMNS + FILTER_ONLY_COLUMNS].where(
    pd.notnull(points_all[DISPLAY_COLUMNS + FILTER_ONLY_COLUMNS]), None
)
records = data_df.to_dict(orient="records")
data_json = json.dumps(records)

column_defs = [{"title": col.replace("_", " ").title(), "field": col, "sorter": "string"} for col in DISPLAY_COLUMNS]
# Numeric columns get a number sorter instead.
NUMERIC_COLUMNS = {
    "ch4_percentage_in_gas_8h", "ch4_percentage_in_gas_24h",
    "methane_intensity", "tddm", "lmf_category_rank", "quartile_rank"
}
for c in column_defs:
    if c["field"] in NUMERIC_COLUMNS:
        c["sorter"] = "number"

columns_json = json.dumps(column_defs)


def options_json(options):
    return json.dumps(options)


html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Accession Filter</title>
  <link href="https://unpkg.com/tabulator-tables@5.5.2/dist/css/tabulator.min.css" rel="stylesheet">
  <script src="https://unpkg.com/tabulator-tables@5.5.2/dist/js/tabulator.min.js"></script>
  <style>
    * {{
      box-sizing: border-box;
    }}
    body {{
      font-family: 'Segoe UI', -apple-system, system-ui, Roboto, sans-serif;
      background: #f6f7fb;
      padding: 24px 16px;
      color: #2b2b33;
    }}
    .card {{
      background: white;
      border-radius: 14px;
      box-shadow: 0 2px 12px rgba(20, 20, 43, 0.06);
      padding: 24px 28px 28px;
      max-width: 1700px;
      margin: 0 auto;
    }}
    h2 {{
      margin: 0 0 4px;
      font-size: 22px;
      font-weight: 700;
      color: #1f1f2e;
    }}
    #meta {{
      color: #8a8a99;
      margin-bottom: 18px;
      font-size: 14px;
    }}
    #meta b {{
      color: #4a4a5a;
    }}

    #filters {{
      display: flex;
      flex-wrap: wrap;
      gap: 18px;
      margin-bottom: 18px;
      align-items: flex-end;
    }}
    .filter-group {{
      display: flex;
      flex-direction: column;
      gap: 4px;
    }}
    .filter-group label {{
      font-size: 12.5px;
      font-weight: 600;
      color: #6a6a78;
    }}
    .filter-group select {{
      padding: 8px 10px;
      border: 1px solid #e2e2ec;
      border-radius: 8px;
      font-size: 14px;
      min-width: 180px;
      background: white;
    }}

    #toolbar {{
      display: flex;
      gap: 10px;
      margin-bottom: 18px;
    }}
    #toolbar button {{
      padding: 9px 18px;
      font-size: 14px;
      font-weight: 600;
      border: none;
      border-radius: 8px;
      cursor: pointer;
    }}
    #btn-download-csv {{
      background: #4C6FFF;
      color: white;
      box-shadow: 0 2px 8px rgba(76, 111, 255, 0.35);
    }}
    #btn-download-csv:hover {{
      background: #3d5ce0;
    }}
    #btn-clear-filters {{
      background: #f1f1f5;
      color: #55555f;
    }}
    #btn-clear-filters:hover {{
      background: #e6e6ec;
    }}

    .tabulator {{
      border: none !important;
      background: transparent !important;
      font-size: 13.5px;
    }}
    .tabulator-header {{
      background: #fafafe !important;
      border-bottom: 2px solid #ececf3 !important;
    }}
    .tabulator-col {{
      background: #fafafe !important;
      border-right: none !important;
    }}
    .tabulator-col-title {{
      color: #3a3a45;
      font-weight: 600;
    }}
    .tabulator-row {{
      background: white !important;
      border-bottom: 1px solid #f0f0f5 !important;
    }}
    .tabulator-row:hover {{
      background: #f5f7ff !important;
    }}
    .tabulator-row.tabulator-row-even {{
      background: #fbfbfe !important;
    }}
    .tabulator-cell {{
      padding: 8px 10px !important;
    }}
    .tabulator-footer {{
      background: #fafafe !important;
      border-top: 2px solid #ececf3 !important;
    }}
    .tabulator-page.active {{
      background: #4C6FFF !important;
      color: white !important;
      border-color: #4C6FFF !important;
    }}
  </style>
</head>
<body>

  <div class="card">
    <h2>Accession Filter</h2>
    <div id="meta">Showing <b><span id="row-count">0</span></b> of <b>{len(records)}</b> accessions.</div>

    <div id="filters">
      <div class="filter-group">
        <label for="filter-group-select">Functional group</label>
        <select id="filter-group-select"></select>
      </div>
      <div class="filter-group">
        <label for="filter-category-select">LMF category</label>
        <select id="filter-category-select"></select>
      </div>
      <div class="filter-group">
        <label for="filter-quartile-select">Quartile</label>
        <select id="filter-quartile-select"></select>
      </div>
    </div>

    <div id="toolbar">
      <button id="btn-download-csv">⬇ Download filtered accessions (CSV)</button>
      <button id="btn-clear-filters">✕ Clear filters</button>
    </div>

    <div id="table"></div>
  </div>

  <script type="text/javascript">
    const tableData = {data_json};
    const columnDefs = {columns_json};

    const groupOptions = {options_json(functional_group_options)};
    const categoryOptions = {options_json(lmf_category_options)};
    const quartileOptions = {options_json(quartile_options)};

    function populateSelect(selectEl, options) {{
      options.forEach(function(opt) {{
        const o = document.createElement("option");
        o.value = opt;
        o.textContent = opt;
        selectEl.appendChild(o);
      }});
    }}

    const groupSelect = document.getElementById("filter-group-select");
    const categorySelect = document.getElementById("filter-category-select");
    const quartileSelect = document.getElementById("filter-quartile-select");

    populateSelect(groupSelect, groupOptions);
    populateSelect(categorySelect, categoryOptions);
    populateSelect(quartileSelect, quartileOptions);

    const table = new Tabulator("#table", {{
      data: tableData,
      columns: columnDefs,
      layout: "fitColumns",
      height: "600px",
      pagination: true,
      paginationSize: 25,
      paginationSizeSelector: [10, 25, 50, 100, true],
    }});

    function updateRowCount() {{
      document.getElementById("row-count").textContent = table.getDataCount("active");
    }}

    function applyFilters() {{
      const filters = [];
      if (groupSelect.value !== "All") {{
        filters.push({{ field: "functional_group", type: "=", value: groupSelect.value }});
      }}
      if (categorySelect.value !== "All") {{
        filters.push({{ field: "lmf_category", type: "=", value: categorySelect.value }});
      }}
      if (quartileSelect.value !== "All") {{
        filters.push({{ field: "quartile", type: "=", value: quartileSelect.value }});
      }}
      table.setFilter(filters);
    }}

    groupSelect.addEventListener("change", applyFilters);
    categorySelect.addEventListener("change", applyFilters);
    quartileSelect.addEventListener("change", applyFilters);

    table.on("tableBuilt", updateRowCount);
    table.on("dataFiltered", updateRowCount);

    // Build the CSV ourselves from the currently filtered ("active")
    // rows, using only the display columns (not the hidden
    // functional_group filter field).
    function toCsv(rows, columns) {{
      const fields = columns.map(c => c.field);
      const escape = (val) => {{
        if (val === null || val === undefined) return "";
        const s = String(val);
        if (/[",\\n]/.test(s)) {{
          return '"' + s.replace(/"/g, '""') + '"';
        }}
        return s;
      }};
      const header = fields.map(escape).join(",");
      const lines = rows.map(row => fields.map(f => escape(row[f])).join(","));
      return [header, ...lines].join("\\n");
    }}

    document.getElementById("btn-download-csv").addEventListener("click", function() {{
      const activeRows = table.getData("active");
      const csv = toCsv(activeRows, columnDefs);
      const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
      const url = URL.createObjectURL(blob);
      const link = document.createElement("a");
      link.href = url;
      link.download = "filtered_accessions.csv";
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      URL.revokeObjectURL(url);
    }});

    document.getElementById("btn-clear-filters").addEventListener("click", function() {{
      groupSelect.value = "All";
      categorySelect.value = "All";
      quartileSelect.value = "All";
      table.setFilter([]);
    }});
  </script>
</body>
</html>
"""

with open("accession_filter.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: accession_filter.html")

Done: accession_filter.html
